In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score,
    roc_curve
)
import matplotlib.pyplot as plt
import os
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer, balanced_accuracy_score


# ── 1. Load data ─────────────────────────────────────────────────
df_bind = pd.read_excel('../data/single_metal_atoms_on_graphene_binding_energy_and_diffusion_barrier.xlsx')
df_bind_g = df_bind.groupby('Metal')[['Binding', 'Diffusion']].mean().reset_index()

df_aff = pd.read_excel("../data/supported_metal_M_oxygen_affinity_QMO_and_support_metal_affinity_QMM_prime.xlsx")
df_aff = df_aff.rename(columns={df_aff.columns[1]: 'QMO'})

df_nps = pd.read_excel('./NPs.xlsx')
df_nps = df_nps.rename(columns={'反应后MOF表面是否有纳米粒子': 'ExternalNP'})
df_nps['MOF'] = df_nps['MOF'].ffill()

# Define your renaming dictionary
mof_rename_dict = {
}

# Apply the replacement
df_nps["MOF"] = df_nps["MOF"].replace(mof_rename_dict)

df_mof = pd.read_csv('../data/MOF_factor.csv').reset_index(drop = True)
# Optional: check for consistency with df_mof
for i in df_nps["MOF"]:
    assert i in list(df_mof["MOF"]), f"{i} not in df_mof"
# ── 2. Merge all descriptors ─────────────────────────────────────
df = (
    df_nps
    .merge(df_mof, on='MOF', how='left')
    .merge(df_bind_g.rename(columns={'Binding': 'BindingEnergy', 'Diffusion': 'DiffusionBarrier'}),
           left_on='M', right_on='Metal', how='left')
    .merge(df_aff[['Metal', 'QMO']], left_on='M', right_on='Metal', how='left')
    .drop(columns=['Metal', 'Metal_aff'], errors='ignore')
)

df = df.dropna(subset=['BindingEnergy', 'DiffusionBarrier', 'QMO'])
df = df.drop(columns=['Metal_x', 'Metal_y'], errors='ignore')

noble_set = {'Au', 'Ag', 'Pt', 'Pd', 'Ir', 'Rh', 'Ru'}
df['Noble'] = df['M'].apply(lambda x: 1 if x in noble_set else 0)

core_feats = ['BindingEnergy','DiffusionBarrier','QMO','Noble']
extra_feats = [f"Factor{i}" for i in range(1, 46)]

features_all = core_feats + extra_feats

# Drop rows with missing values
df = df.dropna(subset=features_all + ['ExternalNP'])

X = df[features_all].copy()
y = df['ExternalNP'].astype(int).copy()

# ── 2. Train & evaluate logistic ────────────────────────────────────────────
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logistic', LogisticRegression(random_state = 42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
acc_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')
bal_acc_scores = cross_val_score(
    pipeline, X, y, cv=cv,
    scoring=make_scorer(balanced_accuracy_score)
)

print(f"5-fold Accuracy: {acc_scores.mean():.3f} ± {acc_scores.std():.3f}")
print(f"5-fold Balanced Accuracy: {bal_acc_scores.mean():.3f} ± {bal_acc_scores.std():.3f}")


# ── 2. Train & evaluate MLP (8-4) ────────────────────────────────────────────
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(hidden_layer_sizes=(10,4), max_iter=100000, random_state=42, tol = 1e-6, activation="tanh"))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
acc_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')
bal_acc_scores = cross_val_score(
    pipeline, X, y, cv=cv,
    scoring=make_scorer(balanced_accuracy_score)
)

print(f"5-fold Accuracy: {acc_scores.mean():.3f} ± {acc_scores.std():.3f}")
print(f"5-fold Balanced Accuracy: {bal_acc_scores.mean():.3f} ± {bal_acc_scores.std():.3f}")


pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(hidden_layer_sizes=(10,4), max_iter=100000, random_state=42, tol = 1e-6, activation="tanh"))
])
# Re-fit on full dataset
pipeline.fit(X, y)

5-fold Accuracy: 0.818 ± 0.081
5-fold Balanced Accuracy: 0.817 ± 0.076
5-fold Accuracy: 0.782 ± 0.067
5-fold Balanced Accuracy: 0.776 ± 0.070


Pipeline(steps=[('scaler', StandardScaler()),
                ('mlp',
                 MLPClassifier(activation='tanh', hidden_layer_sizes=(10, 4),
                               max_iter=100000, random_state=42, tol=1e-06))])

In [ ]:
import itertools
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

# --- core and candidate feature sets ---
core_feats = ['BindingEnergy','DiffusionBarrier','QMO','Noble']
extra_feats = [f"Factor{i}" for i in range(1, 46)]

# --- define models ---
models = {
    "MLP": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=(10,4), activation="tanh",
                              max_iter=100000, random_state=42, tol=1e-6))
    ]),
    "LogReg": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=100000, random_state=42))
    ]),
}

# --- config ---
k = 1  # number of extra features to add to the chosen core subset
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'acc': 'accuracy', 'bal_acc': 'balanced_accuracy'}

# Ensure output dir
out_dir = Path("./results_core_combos")
out_dir.mkdir(parents=True, exist_ok=True)

def _safe_name(parts):
    return "_".join(parts).replace("/", "_")

all_rows = []  # optional: collect everything

# Iterate core subset sizes 2, 3, 4
for core_size in [2, 3, 4]:
    for core_subset in itertools.combinations(core_feats, core_size):
        core_subset = list(core_subset)
        subset_name = _safe_name(core_subset)

        # --- compute per-model baseline on this core subset ---
        baselines = {}
        X_core = X[core_subset]
        for model_name, model in models.items():
            m = clone(model)
            cv_res = cross_validate(
                m, X_core, y, cv=cv, scoring=scoring,
                n_jobs=-1, return_train_score=False
            )
            baselines[model_name] = {
                'baseline_acc_mean':    float(np.mean(cv_res['test_acc'])),
                'baseline_acc_std':     float(np.std(cv_res['test_acc'], ddof=1)),
                'baseline_balacc_mean': float(np.mean(cv_res['test_bal_acc'])),
                'baseline_balacc_std':  float(np.std(cv_res['test_bal_acc'], ddof=1)),
            }

        rows = []  # rows for this core_subset only

        # --- iterate extras and evaluate deltas vs. baseline ---
        for model_name, model in models.items():
            base = baselines[model_name]
            for extras in itertools.combinations(extra_feats, k):
                feats = core_subset + list(extras)

                # skip if any feature missing
                missing = [c for c in feats if c not in X.columns]
                if missing:
                    print(f"[SKIP] Missing features for {model_name} / core={core_subset} / extras={extras}: {missing}")
                    continue

                X_sub = X[feats]

                # --- fit once for "perfect match" check on full data (not CV) ---
                m_fit = clone(model).fit(X_sub, y)
                preds = m_fit.predict(X_sub)
                zero_sum = int(np.sum((preds - y) != 0))
                perfect_match = int(np.array_equal(preds, y))

                # --- cross-validated metrics (accuracy & balanced accuracy) ---
                cv_res = cross_validate(
                    clone(model), X_sub, y,
                    cv=cv, scoring=scoring,
                    n_jobs=-1, return_train_score=False
                )
                acc_mean = float(np.mean(cv_res['test_acc']))
                acc_std  = float(np.std(cv_res['test_acc'], ddof=1))
                bal_mean = float(np.mean(cv_res['test_bal_acc']))
                bal_std  = float(np.std(cv_res['test_bal_acc'], ddof=1))

                res_temp = {
                    "core_subset": tuple(core_subset),
                    "core_size": core_size,
                    "model": model_name,
                    "extras": extras,
                    "n_extras": k,

                    # full-fit snapshot
                    "zero_sum": zero_sum,
                    "perfect_match": perfect_match,

                    # CV metrics (with core+extras)
                    "cv_acc_mean": acc_mean,
                    "cv_acc_std": acc_std,
                    "cv_balacc_mean": bal_mean,
                    "cv_balacc_std": bal_std,

                    # baselines (core only)
                    "baseline_acc_mean": baselines[model_name]['baseline_acc_mean'],
                    "baseline_acc_std":  baselines[model_name]['baseline_acc_std'],
                    "baseline_balacc_mean": baselines[model_name]['baseline_balacc_mean'],
                    "baseline_balacc_std":  baselines[model_name]['baseline_balacc_std'],

                    # deltas (improvements; positive = better)
                    "delta_acc":    acc_mean - baselines[model_name]['baseline_acc_mean'],
                    "delta_balacc": bal_mean - baselines[model_name]['baseline_balacc_mean'],
                }

                rows.append(res_temp)
                all_rows.append(res_temp)

                if perfect_match:
                    print(f"{model_name} core={core_subset} extras={list(extras)} PERFECT MATCH")
                    print(res_temp)

        # --- export CSV for this core subset ---
        results_subset = pd.DataFrame(rows).sort_values(
            ["cv_balacc_mean", "cv_acc_mean"], ascending=False
        ).reset_index(drop=True)
        out_path = out_dir / f"results_core_{subset_name}_k={k}.csv"
        results_subset.to_csv(out_path, index=False)
        print(f"[WROTE] {out_path}  ({len(results_subset)} rows)")

# --- optional: export a master CSV with everything ---
results_all = pd.DataFrame(all_rows).sort_values(
    ["core_size", "cv_balacc_mean", "cv_acc_mean"], ascending=[True, False, False]
).reset_index(drop=True)
results_all.to_csv(out_dir / f"results_all_core_sizes_2to4_k={k}.csv", index=False)
print(f"[WROTE] {out_dir / f'results_all_core_sizes_2to4_k={k}.csv'}  ({len(results_all)} rows)")

In [2]:
results_all = pd.read_csv("./results_core_combos/results_all_core_sizes_2to4_k=1.csv")

In [3]:
results_all["extras"]

0      ('Factor14',)
1      ('Factor26',)
2      ('Factor37',)
3      ('Factor39',)
4       ('Factor4',)
           ...      
985     ('Factor6',)
986    ('Factor28',)
987    ('Factor29',)
988    ('Factor12',)
989    ('Factor23',)
Name: extras, Length: 990, dtype: object

In [4]:
results_all = results_all.sort_values("cv_acc_mean", ascending=False).reset_index(drop=True)

In [5]:
results_all[results_all["perfect_match"] == 1]

,core_subset,core_size,model,extras,n_extras,zero_sum,perfect_match,cv_acc_mean,cv_acc_std,cv_balacc_mean,cv_balacc_std,baseline_acc_mean,baseline_acc_std,baseline_balacc_mean,baseline_balacc_std,delta_acc,delta_balacc
2,"('BindingEnergy', 'DiffusionBarrier', 'Noble')",3,MLP,"('Factor19',)",1,0,1,0.890909,0.040656,0.881667,0.040616,0.845455,0.068935,0.842650,0.067192,0.045455,0.039017
4,"('QMO', 'Noble')",2,MLP,"('Factor39',)",1,0,1,0.881818,0.051826,0.872949,0.058821,0.845455,0.068935,0.846068,0.067034,0.036364,0.026880
6,"('DiffusionBarrier', 'QMO', 'Noble')",3,MLP,"('Factor37',)",1,0,1,0.872727,0.049793,0.869316,0.050413,0.845455,0.068935,0.846068,0.067034,0.027273,0.023248
10,"('DiffusionBarrier', 'QMO', 'Noble')",3,MLP,"('Factor10',)",1,0,1,0.872727,0.049793,0.874872,0.047878,0.845455,0.068935,0.846068,0.067034,0.027273,0.028803
13,"('BindingEnergy', 'DiffusionBarrier', 'QMO', '...",4,MLP,"('Factor37',)",1,0,1,0.872727,0.038030,0.868034,0.037202,0.863636,0.085038,0.864872,0.080414,0.009091,0.003162
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
782,"('BindingEnergy', 'Noble')",2,MLP,"('Factor6',)",1,0,1,0.745455,0.024896,0.727564,0.021747,0.836364,0.060984,0.836068,0.056821,-0.090909,-0.108504
790,"('DiffusionBarrier', 'Noble')",2,MLP,"('Factor43',)",1,0,1,0.745455,0.088607,0.736239,0.078926,0.827273,0.113181,0.834658,0.096924,-0.081818,-0.098419
793,"('BindingEnergy', 'Noble')",2,MLP,"('Factor32',)",1,0,1,0.745455,0.094257,0.747137,0.096456,0.836364,0.060984,0.836068,0.056821,-0.090909,-0.088932
804,"('BindingEnergy', 'Noble')",2,MLP,"('Factor28',)",1,0,1,0.736364,0.038030,0.737051,0.037810,0.836364,0.060984,0.836068,0.056821,-0.100000,-0.099017


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score,
    roc_curve
)
import matplotlib.pyplot as plt

# ── 1. Load data ─────────────────────────────────────────────────
df_bind = pd.read_excel('../data/single_metal_atoms_on_graphene_binding_energy_and_diffusion_barrier.xlsx')
df_bind_g = df_bind.groupby('Metal')[['Binding', 'Diffusion']].mean().reset_index()

df_aff = pd.read_excel("../data/supported_metal_M_oxygen_affinity_QMO_and_support_metal_affinity_QMM_prime.xlsx")
df_aff = df_aff.rename(columns={df_aff.columns[1]: 'QMO'})

df_nps = pd.read_excel('../data/all_NPs.xlsx')
df_nps = df_nps.rename(columns={'反应后MOF表面是否有纳米粒子': 'ExternalNP'})
df_nps['MOF'] = df_nps['MOF'].ffill()

# Define your renaming dictionary
mof_rename_dict = {
}

# Apply the replacement
df_nps["MOF"] = df_nps["MOF"].replace(mof_rename_dict)

df_mof = pd.read_csv('../data/MOF_factor.csv').reset_index(drop = True)
# Optional: check for consistency with df_mof
for i in df_nps["MOF"]:
    assert i in list(df_mof["MOF"]), f"{i} not in df_mof"
# ── 2. Merge all descriptors ─────────────────────────────────────
df = (
    df_nps
    .merge(df_mof, on='MOF', how='left')
    .merge(df_bind_g.rename(columns={'Binding': 'BindingEnergy', 'Diffusion': 'DiffusionBarrier'}),
           left_on='M', right_on='Metal', how='left')
    .merge(df_aff[['Metal', 'QMO']], left_on='M', right_on='Metal', how='left')
    .drop(columns=['Metal', 'Metal_aff'], errors='ignore')
)

df = df.dropna(subset=['BindingEnergy', 'DiffusionBarrier', 'QMO'])
df = df.drop(columns=['Metal_x', 'Metal_y'], errors='ignore')

noble_set = {'Au', 'Ag', 'Pt', 'Pd', 'Ir', 'Rh', 'Ru'}
df['Noble'] = df['M'].apply(lambda x: 1 if x in noble_set else 0)

In [7]:
def safe_parse_tuple(x):
    """Convert a tuple-like string to a Python list."""
    if pd.isna(x):
        return []
    if isinstance(x, (list, tuple)):
        return list(x)
    if isinstance(x, str):
        try:
            val = ast.literal_eval(x)
            if isinstance(val, (list, tuple)):
                return list(val)
            return [val]
        except Exception:
            return [x]
    return [x]

def safe_to_string(x):
    """Flatten tuple/list to single string."""
    if pd.isna(x):
        return ""
    if isinstance(x, (list, tuple, set)):
        if len(x) == 1:
            return str(list(x)[0])
        else:
            return "_".join(map(str, x))
    if isinstance(x, str):
        try:
            val = ast.literal_eval(x)
            if isinstance(val, (list, tuple, set)):
                if len(val) == 1:
                    return str(list(val)[0])
                else:
                    return "_".join(map(str, val))
        except Exception:
            pass
        return x
    return str(x)

# --- apply conversions ---
results_all["core_subset"] = results_all["core_subset"].apply(safe_parse_tuple)
results_all["extras"] = results_all["extras"].apply(safe_to_string)

In [8]:
import itertools
import pandas as pd
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
# --- define models ---
models = {
    "MLP": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=(10,4), activation="tanh",
                              max_iter=100000, random_state=42, tol=1e-6))
    ]),
    "LogReg": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=100000, random_state=42))
    ]),
}

mask = np.array([not i in ["UiO-66-I", "UiO-66-F"] for i in df["MOF"].values])

rows = []
k = 1   # number of extra features
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for index, best in results_all[results_all["zero_sum"] <=1].iterrows():
    if best["cv_balacc_mean"]>0.85:
        # Pick the best row (highest CV balanced accuracy, tiebreaker = accuracy)
        core_feats = best["core_subset"]
        best_model_name = best["model"]
        best_extras = [best["extras"]]
        best_feats = core_feats + best_extras

        print("\n[INFO] Best config:")
        print(f"  model={best_model_name}, features={best_feats}")
        print(f"  cv_balacc_mean={best['cv_balacc_mean']:.4f}, cv_acc_mean={best['cv_acc_mean']:.4f}")

        # Refit best model on the selected features over the full dataset
        best_model = clone(models[best_model_name])
        best_model.fit(df[mask][best_feats], df[mask]["ExternalNP"].values)

        print("Internal_training")
        print(abs(best_model.predict(df[mask][best_feats]) - df[mask]["ExternalNP"].values).sum())

        print("External test")
        print(abs(best_model.predict(df[~mask][best_feats]) - df[~mask]["ExternalNP"].values).sum())

        print(df[~mask].loc[abs(best_model.predict(df[~mask][best_feats]) - df[~mask]["ExternalNP"].values) == 1])


[INFO] Best config:
  model=MLP, features=['BindingEnergy', 'DiffusionBarrier', 'Noble', 'Factor19']
  cv_balacc_mean=0.8817, cv_acc_mean=0.8909
Internal_training
0
External test
6
          MOF   M  苯乙烯加氢产率  反应后是否有pxrd信号  ExternalNP  反应后形貌是否保持  Unnamed: 0  \
110  UiO-66-F  Ir     98.4             1           0          1           8   
112  UiO-66-F  Rh     90.8             1           1          1           8   
120  UiO-66-I  Ir     92.5             1           0          1           9   
121  UiO-66-I  Pd     32.1             1           0          0           9   
122  UiO-66-I  Rh     50.1             1           0          1           9   
124  UiO-66-I  Pt     92.0             1           0          1           9   

      Factor1   Factor2   Factor3  ...  Factor40  Factor41  Factor42  \
110  0.183640 -0.389901 -0.496793  ...  0.366780 -0.394575 -0.053276   
112  0.183640 -0.389901 -0.496793  ...  0.366780 -0.394575 -0.053276   
120 -0.021655  0.472374  0.305175  ... -1.039728

In [ ]:
"""
[INFO] Best config:
  model=MLP, features=['DiffusionBarrier', 'QMO', 'Noble', 'Factor37']
  cv_balacc_mean=0.8693, cv_acc_mean=0.8727
Internal_training
0
External test
3
"""

In [ ]:
import itertools
import pandas as pd
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
# --- define models ---
models = {
    "MLP": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=(10,4), activation="tanh",
                              max_iter=100000, random_state=42, tol=1e-6))
    ]),
    "LogReg": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=100000, random_state=42))
    ]),
}

mask = np.array([not i in ["UiO-66-I", "UiO-66-F"] for i in df["MOF"].values])

rows = []
k = 1   # number of extra features
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for index, best in results_all[results_all["zero_sum"] <=1].iterrows():
    if best["cv_balacc_mean"]>0.85:
        # Pick the best row (highest CV balanced accuracy, tiebreaker = accuracy)
        core_feats = best["core_subset"]
        best_model_name = best["model"]
        best_extras = [best["extras"]]
        best_feats = core_feats + best_extras

        print("\n[INFO] Best config:")
        print(f"  model={best_model_name}, features={best_feats}")
        print(f"  cv_balacc_mean={best['cv_balacc_mean']:.4f}, cv_acc_mean={best['cv_acc_mean']:.4f}")

        # Refit best model on the selected features over the full dataset
        best_model = clone(models[best_model_name])
        best_model.fit(df[mask][best_feats], df[mask]["ExternalNP"].values)

        print("Internal_training")
        print(abs(best_model.predict(df[mask][best_feats]) - df[mask]["ExternalNP"].values).sum())

        print("External test")
        print(abs(best_model.predict(df[~mask][best_feats]) - df[~mask]["ExternalNP"].values).sum())

        print(df[~mask].loc[abs(best_model.predict(df[~mask][best_feats]) - df[~mask]["ExternalNP"].values) == 1])